In [1]:
import json
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv('./data/02-processed/marvel_rivals_final.csv')

df_model = df.dropna(subset=['performance_score']).copy()

df_model["end_session"] = (df_model["series_categorical"] == "end series").astype(int)


In [2]:
#Making temporary X and y just to have dataframes to stratify, unneccesary columns are not removed here
X = df_model.drop(columns=['series_categorical', 'end_session'])
y = df_model['end_session']
groups = df_model['session_id']

#Making test and train dfs stratified by gameplay session
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
train_df = df_model.iloc[train_idx]
test_df = df_model.iloc[test_idx]

#making sure stratification worked
print(f"Total sessions: {df['session_id'].nunique()}")
print(f"Train sessions: {train_df['session_id'].nunique()}")
print(f"Test sessions: {test_df['session_id'].nunique()}")

KeyError: 'session_id'

In [21]:
#NANs in the performance_score column, about 6% of total matches, you choose what to do with them
numerical_features = ['match_duration', 'SR_change', 'performance_score', 'loss_streak']
categorical_features = ['is_win']

feature_cols = numerical_features + categorical_features

X_train = train_df[feature_cols]
y_train = train_df['end_session']

X_test = test_df[feature_cols]
y_test = test_df['end_session']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

In [22]:
#trying to predict series_categorical with ^ variables, implementation up to you but I was gonna use multinomial regression

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
#plot confusion matrix to visualize performance of classifier
cm = confusion_matrix(y_test, y_pred)
cm_percent = cm / cm.sum(axis=1, keepdims=True)

labels = np.array([
    [f"{cm[i,j]}\n({cm_percent[i,j]*100:.1f}%)" for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
])

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=1
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Counts + %)")

plt.xticks([0.5, 1.5], ["Continue", "End"])
plt.yticks([0.5, 1.5], ["Continue", "End"], rotation=0)

plt.tight_layout()
plt.show()